In [ ]:
%pip install numpy pandas xlrd

# Data exploration

In [1]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_excel('./datas/LBV.xls')

In [19]:
df.head()

,Fuel,Oxidizer,Tu (K),P (atm),Equivalence Ratio,Su0 (cm/s),Method,Reference,Note
0,CH4,Air,298,1.0,0.699,15.8,Stagnation,"C. M. Vagelopoulos, F. N. Egolfopoulos, Procee...",Extracted from graph
1,CH4,Air,298,1.0,0.750,18.1,Stagnation,"C. M. Vagelopoulos, F. N. Egolfopoulos, Procee...",Extracted from graph
2,CH4,Air,298,1.0,0.804,24.2,Stagnation,"C. M. Vagelopoulos, F. N. Egolfopoulos, Procee...",Extracted from graph
3,CH4,Air,298,1.0,0.893,31.3,Stagnation,"C. M. Vagelopoulos, F. N. Egolfopoulos, Procee...",Extracted from graph
4,CH4,Air,298,1.0,1.050,37.2,Stagnation,"C. M. Vagelopoulos, F. N. Egolfopoulos, Procee...",Extracted from graph


In [20]:
df = df.drop(columns=['Method','Reference','Note','Fuel'])

In [25]:
df.head()

,Oxidizer,Tu (K),P (atm),Equivalence Ratio,Su0 (cm/s)
0,Air,298,1.0,0.699,15.8
1,Air,298,1.0,0.750,18.1
2,Air,298,1.0,0.804,24.2
3,Air,298,1.0,0.893,31.3
4,Air,298,1.0,1.050,37.2


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 324 entries, 0 to 323
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Oxidizer           324 non-null    object 
 1   Tu (K)             324 non-null    int64  
 2   P (atm)            324 non-null    float64
 3   Equivalence Ratio  324 non-null    float64
 4   Su0 (cm/s)         324 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 12.8+ KB


In [22]:
df.describe()

,Tu (K),P (atm),Equivalence Ratio,Su0 (cm/s)
count,324.000000,324.000000,324.000000,324.000000
mean,309.456790,5.310822,1.013468,24.581200
std,26.150694,10.269946,0.247781,13.671879
min,298.000000,0.250000,0.388593,2.062500
25%,300.000000,1.000000,0.800000,14.008950
50%,300.000000,1.000000,1.000000,22.769050
75%,300.000000,4.934600,1.200000,33.573050
max,400.000000,60.000000,1.796960,72.545200


In [23]:
df.describe(include='object')

,Oxidizer
count,324
unique,6
top,Air
freq,223


In [24]:
df['Oxidizer'].value_counts()

Oxidizer
Air                         223
21 mol %O2 : 79 mol% N2      58
17 mol %O2 : 83 mol% He      19
21 mol %O2 : 79 mol% Ar      13
15 mol %O2 : 85 mol% He       6
21 mol %O2 : 79 mol% CO2      5
Name: count, dtype: int64

converts Oxidizer labels to one hot enocding ->

In [33]:
df_encoded = pd.get_dummies(df, columns=['Oxidizer'], drop_first=True)

In [34]:
df_encoded.head()

,Tu (K),P (atm),Equivalence Ratio,Su0 (cm/s),Oxidizer_17 mol %O2 : 83 mol% He,Oxidizer_21 mol %O2 : 79 mol% Ar,Oxidizer_21 mol %O2 : 79 mol% CO2,Oxidizer_21 mol %O2 : 79 mol% N2,Oxidizer_Air
0,298,1.0,0.699,15.8,False,False,False,False,True
1,298,1.0,0.750,18.1,False,False,False,False,True
2,298,1.0,0.804,24.2,False,False,False,False,True
3,298,1.0,0.893,31.3,False,False,False,False,True
4,298,1.0,1.050,37.2,False,False,False,False,True


In [69]:
X = df_encoded.drop(columns=['Su0 (cm/s)'])
Y = df_encoded['Su0 (cm/s)']

# Model training

In [38]:
%pip install scikit-learn

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached scikit_learn-1.7.2-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
Using cached scikit_learn-1.7.2-cp312-cp312-win_amd64.whl (8.7 MB)
Using cached joblib-1.5.2-py3-none-any.whl (308 kB)


In [64]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

20% data to test and the rest 80% to train

In [70]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [71]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

LinearRegression model

In [72]:
model = LinearRegression()
model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [54]:
from sklearn.metrics import mean_squared_error, r2_score

In [73]:
y_pred = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

MSE: 130.23240098389672
R²: 0.11996010685298819


In [74]:
y_pred[2], y_test.iloc[2]

(np.float64(2.0426403431341527), np.float64(9.0))

RandomForestRegressor model

In [75]:
from sklearn.ensemble import RandomForestRegressor

In [90]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [91]:
y_pred = rf.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

MSE: 16.277663222963326
R²: 0.8900043852743605


In [139]:
def get_pred_error(model, data, label, show_for_each=False):
    # Ensure label is a numpy array
    label_arr = np.array(label)
    pred = model.predict(data)
    
    # Compute absolute errors
    err_arr = np.abs(pred - label_arr)
    
    if show_for_each:
        for p, l, e in zip(pred, label_arr, err_arr):
            print(f"Prediction: {p}, Label: {l}, Error: {e}")
    
    avg_err = np.mean(err_arr)
    print(f"Average Error: {avg_err}")
    
    # Return total error (sum) if needed
    return err_arr.sum()


In [140]:
get_pred_error(rf, X_test[10:15], y_test[10:15], True)

Prediction: 34.335823, Label: 33.4783, Error: 0.8575230000000005
Prediction: 10.749304200000008, Label: 10.1179, Error: 0.6314042000000075
Prediction: 15.200741999999996, Label: 17.358, Error: 2.157258000000004
Prediction: 11.36900209999999, Label: 7.2477, Error: 4.1213020999999905
Prediction: 28.151687999999986, Label: 27.5912, Error: 0.5604879999999852
Average Error: 1.6655950599999976


np.float64(8.327975299999988)

XGBRegressor model

In [ ]:
%pip install xgboost

In [98]:
from xgboost import XGBRegressor

In [114]:
xg = XGBRegressor(
        n_estimators=500,      # more trees usually better for small datasets
        learning_rate=0.05,    # smaller learning rate improves generalization
        max_depth=4,
        random_state=42
    )
xg.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [115]:
y_pred = xg.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R²:", r2_score(y_test, y_pred))

MSE: 11.937399866982883
R²: 0.9193335297205204


In [141]:
get_pred_error(xg, X_test[10:20], y_test[10:20], show_for_each=True)

Prediction: 34.15586853027344, Label: 33.4783, Error: 0.6775685302734402
Prediction: 9.714765548706055, Label: 10.1179, Error: 0.4031344512939459
Prediction: 13.781790733337402, Label: 17.358, Error: 3.576209266662598
Prediction: 8.47150707244873, Label: 7.2477, Error: 1.2238070724487304
Prediction: 28.0832576751709, Label: 27.5912, Error: 0.4920576751708978
Prediction: 18.260900497436523, Label: 18.9601, Error: 0.6991995025634772
Prediction: 16.38348960876465, Label: 13.5429, Error: 2.840589608764649
Prediction: 4.434225559234619, Label: 2.3853, Error: 2.048925559234619
Prediction: 11.17293930053711, Label: 14.4, Error: 3.227060699462891
Prediction: 36.25517654418945, Label: 37.8, Error: 1.544823455810544
Average Error: 1.673337582168579


np.float64(16.73337582168579)

In [151]:
from sklearn.svm import SVR
svr = SVR(kernel="rbf", C=100, epsilon=0.05)

In [152]:
svr.fit(X_train, y_train)

,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,tol,0.001
,C,100
,epsilon,0.05
,shrinking,True
,cache_size,200
,verbose,False
,max_iter,-1


In [153]:
pred = svr.predict(X_test)

In [154]:
get_pred_error(svr, X_test, y_test)

Average Error: 9.869315412185005


np.float64(641.5055017920253)

Try adding more data, feature engineering, Different models & GridSearchCv to improve accuracy...